# KAGGLE HOUSE PRICES: TOP-TIER SOLUTION (FIXED & OPTIMIZED)
**Chiến lược: Advanced Feature Engineering + Skewness Correction + Blended Stacking Ensemble**

Mục tiêu: Đạt thứ hạng cao nhất (Top 1-5%) bằng cách tinh chỉnh đặc trưng và kết hợp mô hình đa dạng.

In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew
from scipy.special import boxcox1p
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.linear_model import Lasso, ElasticNet, Ridge
from sklearn.ensemble import GradientBoostingRegressor, StackingRegressor
from sklearn.svm import SVR
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline

import warnings
warnings.filterwarnings('ignore')
print("Môi trường đã sẵn sàng!")

Môi trường đã sẵn sàng!


## 1. Feature Engineering Bậc Cao

In [17]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# Xóa Outliers theo khuyến nghị chuyên gia
train = train.drop(train[(train['GrLivArea']>4000) & (train['SalePrice']<300000)].index)

y_train = np.log1p(train.SalePrice.values)
train_ID = train['Id']
test_ID = test['Id']
ntrain = train.shape[0]
all_data = pd.concat((train, test)).reset_index(drop=True)
all_data.drop(['SalePrice', 'Id'], axis=1, inplace=True)

## --- Tinh chỉnh dữ liệu thiếu chuyên sâu ---
all_data["LotFrontage"] = all_data.groupby("Neighborhood")["LotFrontage"].transform(lambda x: x.fillna(x.median()))
for col in ('PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'MasVnrType'):
    all_data[col] = all_data[col].fillna('None')
for col in ('GarageYrBlt', 'GarageArea', 'GarageCars', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF','TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea'):
    all_data[col] = all_data[col].fillna(0)
all_data['MSZoning'] = all_data['MSZoning'].fillna(all_data['MSZoning'].mode()[0])
all_data = all_data.drop(['Utilities'], axis=1)
all_data["Functional"] = all_data["Functional"].fillna("Typ")
all_data['Electrical'] = all_data['Electrical'].fillna(all_data['Electrical'].mode()[0])
all_data['KitchenQual'] = all_data['KitchenQual'].fillna(all_data['KitchenQual'].mode()[0])
all_data['Exterior1st'] = all_data['Exterior1st'].fillna(all_data['Exterior1st'].mode()[0])
all_data['Exterior2nd'] = all_data['Exterior2nd'].fillna(all_data['Exterior2nd'].mode()[0])
all_data['SaleType'] = all_data['SaleType'].fillna(all_data['SaleType'].mode()[0])

## --- TẠO ĐẶC TRƯNG MỚI ---
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['TotalBath'] = all_data['FullBath'] + (0.5 * all_data['HalfBath']) + all_data['BsmtFullBath'] + (0.5 * all_data['BsmtHalfBath'])
all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge'] = all_data['YrSold'] - all_data['YearRemodAdd']
all_data['IsNew'] = (all_data['YearBuilt'] == all_data['YrSold']).astype(int)

## --- Xử lý Skewness và Encoding ---
all_data['MSSubClass'] = all_data['MSSubClass'].apply(str)
cols_ord = ('FireplaceQu', 'BsmtQual', 'BsmtCond', 'GarageQual', 'GarageCond', 'ExterQual', 'ExterCond','HeatingQC', 'PoolQC', 'KitchenQual', 'BsmtFinType1', 'BsmtFinType2', 'Functional', 'Fence', 'BsmtExposure', 'GarageFinish', 'LandSlope', 'LotShape', 'PavedDrive', 'Street', 'Alley', 'CentralAir')
for c in cols_ord:
    lbl = LabelEncoder()
    all_data[c] = lbl.fit_transform(list(all_data[c].values))

numeric_feats = all_data.dtypes[all_data.dtypes != "object"].index
skewed_feats = all_data[numeric_feats].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
skewness = pd.DataFrame({'Skew' :skewed_feats})
skewness = skewness[abs(skewness) > 0.75]
for feat in skewness.index:
    all_data[feat] = boxcox1p(all_data[feat], 0.15)

# One-Hot Encoding
all_data = pd.get_dummies(all_data)

# Đảm bảo KHÔNG còn NaN sau khi Encoding (Xử lý lỗi nghiêm trọng)
all_data = all_data.fillna(all_data.median())

train_final = all_data[:ntrain]
test_final = all_data[ntrain:]
print("Xử lý dữ liệu hoàn tất. Shape:", train_final.shape)

Xử lý dữ liệu hoàn tất. Shape: (1458, 239)


## 2. Xây dựng Siêu Mô Hình (Stacking + Blending)

In [18]:
kf = KFold(5, shuffle=True, random_state=42)

lasso = make_pipeline(RobustScaler(), Lasso(alpha=0.0005, random_state=1))
enet = make_pipeline(RobustScaler(), ElasticNet(alpha=0.0005, l1_ratio=.9, random_state=3))
krr = Ridge(alpha=0.6)
svr = make_pipeline(RobustScaler(), SVR(C=20, epsilon=0.008, gamma=0.0003))

gbm = GradientBoostingRegressor(n_estimators=3000, learning_rate=0.05, max_depth=4, max_features='sqrt', min_samples_leaf=15, min_samples_split=10, loss='huber', random_state=5)
model_xgb = xgb.XGBRegressor(colsample_bytree=0.4603, gamma=0.0468, learning_rate=0.05, max_depth=3, min_child_weight=1.7817, n_estimators=2200, reg_alpha=0.4640, reg_lambda=0.8571, subsample=0.5213, random_state=7)
model_lgb = lgb.LGBMRegressor(objective='regression', num_leaves=5, learning_rate=0.05, n_estimators=720, max_bin=55, bagging_fraction=0.8, bagging_freq=5, feature_fraction=0.2319, verbose=-1, random_state=42)
model_cat = CatBoostRegressor(iterations=2000, learning_rate=0.01, depth=4, l2_leaf_reg=3, loss_function='RMSE', verbose=0, random_state=42)

stack_gen = StackingRegressor(
    estimators=[('lasso', lasso), ('enet', enet), ('krr', krr), ('svr', svr), ('gbm', gbm)],
    final_estimator=lasso,
    cv=kf
)

print("Đang huấn luyện dàn siêu mô hình...")
stack_gen.fit(train_final.values, y_train)
model_xgb.fit(train_final.values, y_train)
model_lgb.fit(train_final.values, y_train)
model_cat.fit(train_final.values, y_train)
print("Huấn luyện hoàn tất!")

Đang huấn luyện dàn siêu mô hình...
Huấn luyện hoàn tất!


## 3. Siêu Pha Trộn và Nộp bài

In [19]:
def blend_predictions(X):
    return ((0.40 * stack_gen.predict(X)) + \
            (0.20 * model_xgb.predict(X)) + \
            (0.20 * model_lgb.predict(X)) + \
            (0.20 * model_cat.predict(X)))

final_preds = np.expm1(blend_predictions(test_final.values))
sub = pd.DataFrame({'Id': test_ID, 'SalePrice': final_preds})
sub.to_csv('submission_Grandmaster_Final.csv', index=False)
print("Đã tạo file nộp bài thành công: submission_Grandmaster_Final.csv")
sub.head()

Đã tạo file nộp bài thành công: submission_Grandmaster_Final.csv


,Id,SalePrice
0,1461,121680.074983
1,1462,158703.789046
2,1463,185385.205216
3,1464,195590.399361
4,1465,190662.353904
